# URJA AI - Nepal Electricity Load Forecasting

This notebook recreates and documents the full retraining workflow for URJA AI.
It includes data checks, model evaluation, forecasts, and embedded project visuals.

In [ ]:
import json
from datetime import datetime
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Image, Markdown

from sklearn.ensemble import GradientBoostingRegressor
from sklearn.model_selection import TimeSeriesSplit, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

In [ ]:
BASE_DIR = Path.cwd()
if not (BASE_DIR / 'data').exists() and (BASE_DIR / 'URJA-AI' / 'data').exists():
    BASE_DIR = BASE_DIR / 'URJA-AI'

DATA_PATH = BASE_DIR / 'data' / 'nepal_electricity_demand_extended.csv'
if not DATA_PATH.exists():
    raise FileNotFoundError(f'Dataset not found: {DATA_PATH}')

_tmp_df = pd.read_csv(DATA_PATH, parse_dates=['date'])
latest_year = int(_tmp_df['date'].max().year)
MODEL_DIR = BASE_DIR / 'models' / f'nepal_{latest_year}'
MODEL_DIR.mkdir(parents=True, exist_ok=True)

print('BASE_DIR:', BASE_DIR)
print('DATA_PATH exists:', DATA_PATH.exists())
print('MODEL_DIR:', MODEL_DIR)

## 1) Load and Inspect Dataset

In [ ]:
if not DATA_PATH.exists():
    raise FileNotFoundError(f'Dataset not found: {DATA_PATH}')

df = pd.read_csv(DATA_PATH, parse_dates=['date']).sort_values('date').reset_index(drop=True)
print('Rows:', len(df))
print('Date range:', df['date'].min().date(), 'to', df['date'].max().date())
print('Fiscal years:', df['fiscal_year'].nunique())

display(df.head(5))
display(df.tail(5))

In [ ]:
plt.figure(figsize=(12, 4))
plt.plot(df['date'], df['demand_gwh'], linewidth=2)
plt.title('Nepal Monthly Electricity Demand (GWh)')
plt.xlabel('Date')
plt.ylabel('Demand (GWh)')
plt.grid(alpha=0.25)
plt.tight_layout()
plt.show()

## 2) Feature Engineering

In [ ]:
def create_features(inp: pd.DataFrame) -> pd.DataFrame:
    out = inp.copy()

    out['month'] = out['date'].dt.month
    out['year'] = out['date'].dt.year
    out['quarter'] = out['date'].dt.quarter

    out['month_sin'] = np.sin(2 * np.pi * out['month'] / 12)
    out['month_cos'] = np.cos(2 * np.pi * out['month'] / 12)

    def get_season(month):
        if month in [6, 7, 8, 9]:
            return 1
        elif month in [10, 11]:
            return 2
        elif month in [12, 1, 2]:
            return 3
        else:
            return 4

    out['season'] = out['month'].apply(get_season)
    out['time_idx'] = range(len(out))

    for lag in [1, 2, 3, 6, 12]:
        out[f'lag_{lag}'] = out['demand_gwh'].shift(lag)

    out['rolling_mean_3'] = out['demand_gwh'].shift(1).rolling(window=3).mean()
    out['rolling_mean_6'] = out['demand_gwh'].shift(1).rolling(window=6).mean()
    out['rolling_mean_12'] = out['demand_gwh'].shift(1).rolling(window=12).mean()
    out['rolling_std_6'] = out['demand_gwh'].shift(1).rolling(window=6).std()
    out['rolling_std_12'] = out['demand_gwh'].shift(1).rolling(window=12).std()

    out['yoy_growth'] = (out['demand_gwh'] - out['lag_12']) / out['lag_12']
    out['trend'] = np.arange(len(out))
    out['trend_squared'] = out['trend'] ** 2

    return out

FEATURE_COLS = [
    'month', 'year', 'quarter', 'month_sin', 'month_cos', 'season', 'time_idx',
    'lag_1', 'lag_2', 'lag_3', 'lag_6', 'lag_12',
    'rolling_mean_3', 'rolling_mean_6', 'rolling_mean_12',
    'rolling_std_6', 'rolling_std_12',
    'trend', 'trend_squared'
]

df_features = create_features(df)
df_clean = df_features.dropna().reset_index(drop=True)

X = df_clean[FEATURE_COLS].values
y = df_clean['demand_gwh'].values

scaler_X = StandardScaler()
scaler_y = StandardScaler()
X_scaled = scaler_X.fit_transform(X)
y_scaled = scaler_y.fit_transform(y.reshape(-1, 1)).ravel()

print('Training samples:', len(df_clean))

## 3) Train, Validate, and Evaluate

In [ ]:
tscv = TimeSeriesSplit(n_splits=5)
model = GradientBoostingRegressor(
    n_estimators=200,
    max_depth=5,
    learning_rate=0.1,
    min_samples_split=5,
    min_samples_leaf=3,
    subsample=0.8,
    random_state=42
)

TEST_SIZE = 12
X_train, X_test = X_scaled[:-TEST_SIZE], X_scaled[-TEST_SIZE:]
y_train, y_test = y[:-TEST_SIZE], y[-TEST_SIZE:]
y_train_scaled = y_scaled[:-TEST_SIZE]

cv_scores = cross_val_score(model, X_train, y_train_scaled, cv=tscv, scoring='neg_mean_squared_error')
cv_rmse = np.sqrt(-cv_scores)

model.fit(X_train, y_train_scaled)
y_test_pred_scaled = model.predict(X_test)
y_test_pred = scaler_y.inverse_transform(y_test_pred_scaled.reshape(-1, 1)).ravel()

test_mae = mean_absolute_error(y_test, y_test_pred)
test_rmse = np.sqrt(mean_squared_error(y_test, y_test_pred))
test_r2 = r2_score(y_test, y_test_pred)
test_mape = np.mean(np.abs((y_test - y_test_pred) / y_test)) * 100

metrics_df = pd.DataFrame([
    {'metric': 'CV RMSE (mean)', 'value': float(cv_rmse.mean())},
    {'metric': 'CV RMSE (std)', 'value': float(cv_rmse.std())},
    {'metric': 'Test MAE (GWh)', 'value': float(test_mae)},
    {'metric': 'Test RMSE (GWh)', 'value': float(test_rmse)},
    {'metric': 'Test R2', 'value': float(test_r2)},
    {'metric': 'Test MAPE (%)', 'value': float(test_mape)}
])
display(metrics_df)

In [ ]:
# retrain on full data for production artifacts
model.fit(X_scaled, y_scaled)

joblib.dump(model, MODEL_DIR / 'nepal_load_forecast_model.joblib')
joblib.dump(scaler_X, MODEL_DIR / 'scaler_X.pkl')
joblib.dump(scaler_y, MODEL_DIR / 'scaler_y.pkl')

config = {
    'model_type': 'GradientBoostingRegressor',
    'feature_columns': FEATURE_COLS,
    'training_samples': len(df_clean),
    'date_range': {
        'start': df['date'].min().strftime('%Y-%m-%d'),
        'end': df['date'].max().strftime('%Y-%m-%d')
    },
    'trained_at': datetime.now().strftime('%Y-%m-%d %H:%M:%S'),
    'model_params': {
        'n_estimators': 200,
        'max_depth': 5,
        'learning_rate': 0.1
    }
}

metrics = {
    'mae': float(test_mae),
    'rmse': float(test_rmse),
    'r2': float(test_r2),
    'mape': float(test_mape / 100),
    'cv_rmse_mean': float(cv_rmse.mean()),
    'cv_rmse_std': float(cv_rmse.std()),
    'training_samples': len(df_clean),
    'test_samples': TEST_SIZE
}

(MODEL_DIR / 'config.json').write_text(json.dumps(config, indent=2), encoding='utf-8')
(MODEL_DIR / 'metrics.json').write_text(json.dumps(metrics, indent=2), encoding='utf-8')
print('Saved model artifacts to', MODEL_DIR)

## 4) Generate 12-Month Forecast

In [ ]:
def forecast_next_months(model, scaler_X, scaler_y, df_base, feature_cols, months=12):
    last_date = df_base['date'].iloc[-1]
    demand_history = list(df_base['demand_gwh'].values[-12:])
    last_time_idx = len(df_base) - 1

    timestamps, predictions = [], []

    for i in range(months):
        next_date = last_date + pd.DateOffset(months=i + 1)
        month = next_date.month

        features = {
            'month': month,
            'year': next_date.year,
            'quarter': ((month - 1) // 3) + 1,
            'month_sin': np.sin(2 * np.pi * month / 12),
            'month_cos': np.cos(2 * np.pi * month / 12),
            'season': 1 if month in [6, 7, 8, 9] else (2 if month in [10, 11] else (3 if month in [12, 1, 2] else 4)),
            'time_idx': last_time_idx + i + 1,
            'lag_1': demand_history[-1],
            'lag_2': demand_history[-2],
            'lag_3': demand_history[-3],
            'lag_6': demand_history[-6],
            'lag_12': demand_history[-12] if len(demand_history) >= 12 else demand_history[0],
            'rolling_mean_3': np.mean(demand_history[-3:]),
            'rolling_mean_6': np.mean(demand_history[-6:]),
            'rolling_mean_12': np.mean(demand_history[-12:]),
            'rolling_std_6': np.std(demand_history[-6:]),
            'rolling_std_12': np.std(demand_history[-12:]),
            'trend': last_time_idx + i + 1,
            'trend_squared': (last_time_idx + i + 1) ** 2
        }

        X_new = np.array([[features[col] for col in feature_cols]])
        X_new_scaled = scaler_X.transform(X_new)
        pred_scaled = model.predict(X_new_scaled)
        pred_actual = scaler_y.inverse_transform(pred_scaled.reshape(-1, 1)).ravel()[0]

        timestamps.append(next_date.strftime('%Y-%m'))
        predictions.append(pred_actual)

        demand_history.append(pred_actual)
        demand_history.pop(0)

    return timestamps, predictions

timestamps, predictions = forecast_next_months(model, scaler_X, scaler_y, df, FEATURE_COLS, months=12)
forecast_df = pd.DataFrame({'date': timestamps, 'forecast_gwh': predictions})
forecast_df.to_csv(MODEL_DIR / 'forecast_12months.csv', index=False)

display(forecast_df)
print(f"Annual total forecast: {sum(predictions):,.2f} GWh ({sum(predictions)/1000:.2f} TWh)")

## 5) Code Explanation With Figure Evidence

This section connects each generated figure to the exact part of the pipeline so the visuals are not just decorative.

1. Data behavior plot: confirms trend and seasonality before feature engineering.
2. Training results plot: compares model predictions vs actual values on validation/test periods.
3. Forecast plot: shows the next 12-month projection produced by the trained model.

### Figure A - Demand Analysis (from load + inspect step)

Expected meaning:
- Shows long-term demand growth and monthly seasonality.
- Helps justify lag features and rolling statistics used later.

Image rendering is handled in the next code cell from the active model folder.

### Figure B - Training Results (from train + evaluate step)

Expected meaning:
- Visual check of prediction quality against true demand values.
- Helps explain MAE, RMSE, R2, and MAPE metrics.

Image rendering is handled in the next code cell from the active model folder.

### Figure C - Forecast Plot (from 12-month forecast step)

Expected meaning:
- Final model output for future monthly demand.
- Used to interpret near-term planning implications.

Image rendering is handled in the next code cell from the active model folder.

In [ ]:
figure_notes = {
    'demand_analysis.png': 'Figure A: Demand analysis used for feature design decisions',
    'training_results.png': 'Figure B: Validation/test performance interpretation',
    'forecast_plot.png': 'Figure C: 12-month forecast output'
}

for filename, note in figure_notes.items():
    img = MODEL_DIR / filename
    print(f'{filename} exists = {img.exists()}')
    print(note)
    if img.exists():
        display(Image(filename=str(img)))
    print('-' * 72)

In [ ]:
# Keep app data file synced
import shutil
shutil.copy(DATA_PATH, BASE_DIR / 'data' / 'nepal_electricity_demand.csv')
print('Updated main data file with extended dataset')
print('Notebook recreation complete')